## Hyperparameter tuning

In [ ]:
import wandb

In [ ]:
import os
import gc
import sys
import time

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import pyro
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

In [ ]:
sys.path.append('../Lynx')
sys.path.append('../Lynx/util')
sys.path.append('../Lynx/models')

# LYNX imports
import IO, plot, utils, trajectory
import vgae, configs, dataset
from importlib import reload

### Set wandb configs

In [ ]:
wandb.login()

In [ ]:
sweep_configuration = {
    "method": "grid",
    "name": "sweep",
    "metric": {"goal": "minimize", "name": "val_loss"},
    "parameters": {
        "graph_radius": {"values": [25, 50, 100]},
        "latent_dim": {"values": [4, 6, 8, 10, 12]},
        "lr": {"values": [1e-4, 1e-3, 1e-2]},
        "activation": {"values": ["SiLU", "ReLU", "LeakyReLU"]}
    },
}

In [ ]:
sweep_id = wandb.sweep(sweep=sweep_configuration, project="LYNX")

### Load data

In [ ]:
# Load paired Xenium & DESI
xenium_path = '../liver3d_bucket/Xenium example /processed'
desi_path = '../liver3d_bucket/DESI/processed'
sample_id = 'NIH_F5'

adata_xenium = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi = sc.read_h5ad(os.path.join(desi_path, sample_id+'.h5'))
adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

In [ ]:
if 'cell_type' in adata_xenium.obs.keys():
    adata_xenium.obs['leiden'], categories = adata_xenium.obs.cell_type.factorize()
    categories = categories.values
else:
    adata_norm = adata_xenium.copy()
    sc.pp.normalize_total(adata_norm)
    sc.pp.log1p(adata_norm)

    sc.pp.pca(adata_norm)
    sc.pp.neighbors(adata_norm)
    sc.tl.leiden(adata_norm, random_state=42)
    adata_xenium.obs['leiden'] = adata_norm.obs['leiden'].copy()
    del adata_norm   

### Run Hyperparam Sweep

In [ ]:
def main():
    run = wandb.init()

    #### SET DATASET CONFIGS ####
    n_subgraphs = 16
    k = 100
    r = wandb.config.graph_radius
    sigma = 20
    
    graph_data = dataset.HeteroDataset(
        adatas_ref=adata_xenium, 
        adatas_query=adata_desi,
        n_subgraphs=n_subgraphs, 
        k=k,
        r=r,
        is_weighted=True,
        sigma=sigma,
        verbose=True
    )
    
    train_data, val_data = random_split(graph_data, [0.7, 0.3])
    train_dl, val_dl = DataLoader(train_data, shuffle=True), DataLoader(val_data)

    #### SET MODEL CONFIGS ####

    # Model parameters
    n_hidden = 32
    n_latent = wandb.config.latent_dim
    act_dict = {"SiLU": nn.SiLU(), "ReLU": nn.ReLU(), "LeakyReLU": nn.LeakyReLU()}
    act = act_dict[wandb.config.activation]
    
    # Training parameters
    n_epochs = 500
    lr = wandb.config.lr
    patience = 20
    
    # Training & Inference
    train_configs = configs.set_train_configs(
        n_epochs=n_epochs, lr=lr, patience=patience, 
        device=torch.device('cuda'),
        anneal=False,
        verbose=True,
        #scheduler step and gamma applies gamma every step
        step_size=100, 
        gamma=0.1
    )
    
    model_configs = configs.set_model_configs(
        c_in=adata_xenium.shape[1],   # ref-dim 
        c_aux=adata_desi.shape[1],  # query-dim
        c_hidden=n_hidden, 
        c_latent=n_latent,
        act=act,
        ref=graph_data.ref, 
        query=graph_data.query,
        k_hop=1,
        num_heads=1,
        num_clusters=graph_data.num_clusters,
        verbose=True
    ) 

    # TRAIN
    model = vgae.HeteroVGAE(model_configs, device=torch.device('cuda'))
    model.fit(train_configs, train_dl=train_dl, val_dl=val_dl, DEBUG=True, log_wandb=True)

    del model, graph_data, train_data, val_data, train_dl, val_dl
    pyro.clear_param_store()
    torch.cuda.empty_cache()
    reload(vgae)

In [ ]:
wandb.agent(sweep_id, function=main)